# RQ1 Prompt Comparison Analysis: Old vs New CWE-Based Few-Shot Prompts

**Date**: 2025-11-02  
**Purpose**: Compare vulnerability detection performance between original LLM-generated few-shot examples and new CWE-based canonical examples

## Research Question
Does the quality of few-shot examples (CWE-based vs LLM-generated) affect the Chain-of-Thought (CoT) paradox where few-shot prompts degrade performance?

## Experiments Compared

### Original Prompts (Phase 1 & 2a)
- Location: `results/mars/` (4B) and `results/runpod/` (30B)
- Few-shot examples: LLM-generated vulnerability examples

### New CWE Prompts (Re-run)
- Location: `results/mars_rerun/` (4B) and `results/runpod_rerun/` (30B)
- Few-shot examples: Canonical CWE-based examples
  - CWE-787: Buffer overflow (strcpy)
  - CWE-401: Memory leak (missing delete)
  - CWE-193: Off-by-one error

## Key Questions
1. Does using canonical CWE examples improve F1 scores?
2. Is the CoT paradox (few-shot degrading performance) still present with better prompts?
3. Do Thinking models benefit more from high-quality prompts than Instruct models?
4. Does prompt quality affect energy consumption patterns?

## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Paths
PROJECT_ROOT = Path('/Users/shanetan/Documents/Code_Projects/SMU/SCIS_EngD/agent-green')
RESULTS_DIR = PROJECT_ROOT / 'results'

# Old results (original prompts)
OLD_MARS_DIR = RESULTS_DIR / 'mars'
OLD_RUNPOD_DIR = RESULTS_DIR / 'runpod'

# New results (CWE prompts)
NEW_MARS_DIR = RESULTS_DIR / 'mars_rerun'
NEW_RUNPOD_DIR = RESULTS_DIR / 'runpod_rerun'

# Output directory
OUTPUT_DIR = RESULTS_DIR / 'analysis_prompt_comparison'
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"\nDirectories exist:")
print(f"  Old Mars: {OLD_MARS_DIR.exists()}")
print(f"  Old RunPod: {OLD_RUNPOD_DIR.exists()}")
print(f"  New Mars: {NEW_MARS_DIR.exists()}")
print(f"  New RunPod: {NEW_RUNPOD_DIR.exists()}")

## 2. Load Old Results (Original Prompts)

In [ ]:
def find_summary_file(directory, model_keyword, prompt_type='few'):
    """Find summary vulnerability metrics CSV file."""
    pattern = f"*{prompt_type}*{model_keyword}*summary_vulnerability_metrics.csv"
    files = list(directory.glob(pattern))
    if not files:
        # Try recursive search in subdirectories (for runpod structure)
        files = list(directory.glob(f"**/{pattern}"))
    
    if files:
        return files[0]
    return None

# Load Old Mars 4B results
old_4b_instruct = find_summary_file(OLD_MARS_DIR, 'Instruct')
old_4b_thinking = find_summary_file(OLD_MARS_DIR, 'Thinking')

print("Old 4B Results:")
print(f"  Instruct: {old_4b_instruct}")
print(f"  Thinking: {old_4b_thinking}")

# Load Old RunPod 30B results
old_30b_instruct = find_summary_file(OLD_RUNPOD_DIR, 'Instruct')
old_30b_thinking = find_summary_file(OLD_RUNPOD_DIR, 'Thinking')

print("\nOld 30B Results:")
print(f"  Instruct: {old_30b_instruct}")
print(f"  Thinking: {old_30b_thinking}")

## 3. Load New Results (CWE Prompts)

In [ ]:
# Load New Mars 4B results
new_4b_instruct = find_summary_file(NEW_MARS_DIR, 'Instruct')
new_4b_thinking = find_summary_file(NEW_MARS_DIR, 'Thinking')

print("New 4B Results (CWE Prompts):")
print(f"  Instruct: {new_4b_instruct}")
print(f"  Thinking: {new_4b_thinking}")

# Load New RunPod 30B results
new_30b_instruct = find_summary_file(NEW_RUNPOD_DIR, 'Instruct')
new_30b_thinking = find_summary_file(NEW_RUNPOD_DIR, 'Thinking')

print("\nNew 30B Results (CWE Prompts):")
print(f"  Instruct: {new_30b_instruct}")
print(f"  Thinking: {new_30b_thinking}")

## 4. Parse Metrics from Summary Files

In [ ]:
def load_metrics(filepath):
    """Load metrics from summary CSV file."""
    if filepath is None or not filepath.exists():
        return None
    
    df = pd.read_csv(filepath)
    
    # Extract metrics (row 0 usually has overall metrics)
    if len(df) > 0:
        return {
            'accuracy': df.iloc[0]['accuracy'] if 'accuracy' in df.columns else None,
            'precision': df.iloc[0]['precision'] if 'precision' in df.columns else None,
            'recall': df.iloc[0]['recall'] if 'recall' in df.columns else None,
            'f1': df.iloc[0]['f1-score'] if 'f1-score' in df.columns else None,
        }
    return None

# Create comparison dataframe
comparison_data = []

# 4B Instruct
old_4b_inst_metrics = load_metrics(old_4b_instruct)
new_4b_inst_metrics = load_metrics(new_4b_instruct)
if old_4b_inst_metrics and new_4b_inst_metrics:
    comparison_data.append({
        'Model': '4B',
        'Type': 'Instruct',
        'Prompt': 'Old (LLM)',
        **old_4b_inst_metrics
    })
    comparison_data.append({
        'Model': '4B',
        'Type': 'Instruct',
        'Prompt': 'New (CWE)',
        **new_4b_inst_metrics
    })

# 4B Thinking
old_4b_think_metrics = load_metrics(old_4b_thinking)
new_4b_think_metrics = load_metrics(new_4b_thinking)
if old_4b_think_metrics and new_4b_think_metrics:
    comparison_data.append({
        'Model': '4B',
        'Type': 'Thinking',
        'Prompt': 'Old (LLM)',
        **old_4b_think_metrics
    })
    comparison_data.append({
        'Model': '4B',
        'Type': 'Thinking',
        'Prompt': 'New (CWE)',
        **new_4b_think_metrics
    })

# 30B Instruct
old_30b_inst_metrics = load_metrics(old_30b_instruct)
new_30b_inst_metrics = load_metrics(new_30b_instruct)
if old_30b_inst_metrics and new_30b_inst_metrics:
    comparison_data.append({
        'Model': '30B',
        'Type': 'Instruct',
        'Prompt': 'Old (LLM)',
        **old_30b_inst_metrics
    })
    comparison_data.append({
        'Model': '30B',
        'Type': 'Instruct',
        'Prompt': 'New (CWE)',
        **new_30b_inst_metrics
    })

# 30B Thinking
old_30b_think_metrics = load_metrics(old_30b_thinking)
new_30b_think_metrics = load_metrics(new_30b_thinking)
if old_30b_think_metrics and new_30b_think_metrics:
    comparison_data.append({
        'Model': '30B',
        'Type': 'Thinking',
        'Prompt': 'Old (LLM)',
        **old_30b_think_metrics
    })
    comparison_data.append({
        'Model': '30B',
        'Type': 'Thinking',
        'Prompt': 'New (CWE)',
        **new_30b_think_metrics
    })

df_comparison = pd.DataFrame(comparison_data)
print("\n=== Comparison Data ===")
print(df_comparison.to_string(index=False))

## 5. Calculate Deltas (New - Old)

In [ ]:
# Calculate deltas for each model/type combination
delta_data = []

for model in ['4B', '30B']:
    for model_type in ['Instruct', 'Thinking']:
        old_row = df_comparison[(df_comparison['Model'] == model) & 
                                 (df_comparison['Type'] == model_type) & 
                                 (df_comparison['Prompt'] == 'Old (LLM)')]
        new_row = df_comparison[(df_comparison['Model'] == model) & 
                                 (df_comparison['Type'] == model_type) & 
                                 (df_comparison['Prompt'] == 'New (CWE)')]
        
        if len(old_row) > 0 and len(new_row) > 0:
            delta_data.append({
                'Model': model,
                'Type': model_type,
                'ΔAccuracy': new_row.iloc[0]['accuracy'] - old_row.iloc[0]['accuracy'],
                'ΔPrecision': new_row.iloc[0]['precision'] - old_row.iloc[0]['precision'],
                'ΔRecall': new_row.iloc[0]['recall'] - old_row.iloc[0]['recall'],
                'ΔF1': new_row.iloc[0]['f1'] - old_row.iloc[0]['f1'],
                'Old_F1': old_row.iloc[0]['f1'],
                'New_F1': new_row.iloc[0]['f1']
            })

df_deltas = pd.DataFrame(delta_data)

print("\n=== Delta Analysis (New CWE - Old LLM) ===")
print(df_deltas.to_string(index=False))

# Highlight significant changes (|ΔF1| > 0.02 i.e., 2 percentage points)
print("\n=== Significant Changes (|ΔF1| > 2pp) ===")
significant = df_deltas[abs(df_deltas['ΔF1']) > 0.02]
if len(significant) > 0:
    print(significant[['Model', 'Type', 'Old_F1', 'New_F1', 'ΔF1']].to_string(index=False))
else:
    print("No significant changes detected.")

## 6. Visualization: F1 Score Comparison

In [ ]:
# Create side-by-side bar chart
fig, ax = plt.subplots(figsize=(14, 6))

models = df_comparison['Model'] + ' ' + df_comparison['Type']
x = np.arange(len(df_comparison) // 2)
width = 0.35

# Get old and new F1 scores
old_f1 = df_comparison[df_comparison['Prompt'] == 'Old (LLM)']['f1'].values
new_f1 = df_comparison[df_comparison['Prompt'] == 'New (CWE)']['f1'].values
labels = (df_comparison[df_comparison['Prompt'] == 'Old (LLM)']['Model'] + ' ' + 
          df_comparison[df_comparison['Prompt'] == 'Old (LLM)']['Type']).values

bars1 = ax.bar(x - width/2, old_f1, width, label='Old (LLM-generated)', color='#3498db', alpha=0.8)
bars2 = ax.bar(x + width/2, new_f1, width, label='New (CWE-based)', color='#e74c3c', alpha=0.8)

# Add value labels on bars
for bar in bars1:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.3f}', ha='center', va='bottom', fontsize=9)

for bar in bars2:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.3f}', ha='center', va='bottom', fontsize=9)

ax.set_xlabel('Model Configuration', fontsize=12, fontweight='bold')
ax.set_ylabel('F1 Score', fontsize=12, fontweight='bold')
ax.set_title('F1 Score Comparison: Old LLM-Generated vs New CWE-Based Few-Shot Prompts', 
             fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=0)
ax.legend(loc='upper right', fontsize=11)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 1.0)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'f1_comparison_old_vs_new.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Saved: f1_comparison_old_vs_new.png")

## 7. Visualization: Delta F1 Scores

In [ ]:
# Create delta bar chart
fig, ax = plt.subplots(figsize=(12, 6))

labels = df_deltas['Model'] + ' ' + df_deltas['Type']
deltas = df_deltas['ΔF1'].values
colors = ['#27ae60' if d > 0 else '#e74c3c' for d in deltas]

bars = ax.barh(labels, deltas, color=colors, alpha=0.7)

# Add value labels
for i, (bar, delta) in enumerate(zip(bars, deltas)):
    x_pos = delta + (0.002 if delta > 0 else -0.002)
    ha = 'left' if delta > 0 else 'right'
    ax.text(x_pos, bar.get_y() + bar.get_height()/2,
            f'{delta:+.3f}', ha=ha, va='center', fontsize=10, fontweight='bold')

# Add reference line at 0
ax.axvline(x=0, color='black', linestyle='-', linewidth=0.8)

# Add significance threshold lines at ±2pp
ax.axvline(x=0.02, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
ax.axvline(x=-0.02, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
ax.text(0.02, len(labels)-0.5, '+2pp', fontsize=8, ha='center', color='gray')
ax.text(-0.02, len(labels)-0.5, '-2pp', fontsize=8, ha='center', color='gray')

ax.set_xlabel('ΔF1 Score (New CWE - Old LLM)', fontsize=12, fontweight='bold')
ax.set_ylabel('Model Configuration', fontsize=12, fontweight='bold')
ax.set_title('Impact of CWE-Based Prompts on F1 Score\n(Green = Improvement, Red = Degradation)', 
             fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'delta_f1_scores.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Saved: delta_f1_scores.png")

## 8. Statistical Summary

In [ ]:
print("\n" + "="*80)
print("STATISTICAL SUMMARY: CWE PROMPT IMPACT")
print("="*80)

print(f"\n1. Overall Impact:")
print(f"   Mean ΔF1: {df_deltas['ΔF1'].mean():+.4f}")
print(f"   Median ΔF1: {df_deltas['ΔF1'].median():+.4f}")
print(f"   Std Dev: {df_deltas['ΔF1'].std():.4f}")

print(f"\n2. Improvements vs Degradations:")
improvements = len(df_deltas[df_deltas['ΔF1'] > 0])
degradations = len(df_deltas[df_deltas['ΔF1'] < 0])
no_change = len(df_deltas[df_deltas['ΔF1'] == 0])
print(f"   Improved: {improvements}/{len(df_deltas)} configurations")
print(f"   Degraded: {degradations}/{len(df_deltas)} configurations")
print(f"   No change: {no_change}/{len(df_deltas)} configurations")

print(f"\n3. Significant Changes (|ΔF1| > 2pp):")
significant = df_deltas[abs(df_deltas['ΔF1']) > 0.02]
print(f"   Count: {len(significant)}/{len(df_deltas)} configurations")
if len(significant) > 0:
    for _, row in significant.iterrows():
        direction = "⬆️ IMPROVED" if row['ΔF1'] > 0 else "⬇️ DEGRADED"
        print(f"   - {row['Model']} {row['Type']}: {row['Old_F1']:.3f} → {row['New_F1']:.3f} ({row['ΔF1']:+.3f}) {direction}")

print(f"\n4. Model Size Comparison:")
delta_4b = df_deltas[df_deltas['Model'] == '4B']['ΔF1'].mean()
delta_30b = df_deltas[df_deltas['Model'] == '30B']['ΔF1'].mean()
print(f"   4B models mean ΔF1: {delta_4b:+.4f}")
print(f"   30B models mean ΔF1: {delta_30b:+.4f}")
print(f"   Difference: {abs(delta_30b - delta_4b):.4f}")

print(f"\n5. Instruct vs Thinking:")
delta_instruct = df_deltas[df_deltas['Type'] == 'Instruct']['ΔF1'].mean()
delta_thinking = df_deltas[df_deltas['Type'] == 'Thinking']['ΔF1'].mean()
print(f"   Instruct models mean ΔF1: {delta_instruct:+.4f}")
print(f"   Thinking models mean ΔF1: {delta_thinking:+.4f}")
print(f"   Difference: {abs(delta_thinking - delta_instruct):.4f}")

print("\n" + "="*80)

## 9. Export Results

In [ ]:
# Export comparison table
df_comparison.to_csv(OUTPUT_DIR / 'prompt_comparison_full.csv', index=False)
print("✅ Saved: prompt_comparison_full.csv")

# Export delta table
df_deltas.to_csv(OUTPUT_DIR / 'prompt_comparison_deltas.csv', index=False)
print("✅ Saved: prompt_comparison_deltas.csv")

# Export to Excel with formatting
with pd.ExcelWriter(OUTPUT_DIR / 'prompt_comparison_analysis.xlsx', engine='openpyxl') as writer:
    df_comparison.to_excel(writer, sheet_name='Full Comparison', index=False)
    df_deltas.to_excel(writer, sheet_name='Deltas', index=False)

print("✅ Saved: prompt_comparison_analysis.xlsx")

print(f"\n📁 All outputs saved to: {OUTPUT_DIR}")

## 10. Key Findings & Conclusions

### Summary
[To be filled after running analysis]

### Answer to Research Questions

**Q1: Does using canonical CWE examples improve F1 scores?**
- [To be determined from delta analysis]

**Q2: Is the CoT paradox still present with better prompts?**
- [Compare few-shot vs zero-shot performance with new prompts]

**Q3: Do Thinking models benefit more from high-quality prompts?**
- [Compare ΔF1 for Thinking vs Instruct models]

**Q4: Does prompt quality affect energy consumption patterns?**
- [Requires CodeCarbon analysis - separate notebook section]

### Implications
- [Research implications to be documented]
- [Recommendations for future work]